# Week 8 · Monday: Feature Engineering and Model Evaluation

Last Wednesday's penguins build filled blanks, one-hot encoded and scaled a table, all learned from the training rows. Today goes deeper on those same steps, on one table of 1,000 loan applicants, then chooses how to judge the model. Today:

1. **Feature engineering:** the columns a lender may not use, new features built from old ones, and the split first
2. **Encoding categorical data:** one-hot, ordinal and frequency encoding, and every step in one `Pipeline`
3. **Missing values and outliers:** fill, flag, clip or log, each one measured
4. **Feature selection techniques:** a filter, a model-based ranking, and whether dropping columns helps
5. **Choosing a metric:** accuracy on a 70/30 table, averaged F1, the confusion matrix as costs, and the decision threshold
6. **Avoiding overfitting:** L2 and L1 regularisation in scikit-learn and Keras, and cross-validation
7. **The test set once, and your project:** the kept model scored once, and a checklist for your own notebook

Every model today is judged on the same 200 validation rows, and every result goes into one `results` table, so each change is measured against the one before it. The test set waits until section 7.

Parts marked *Going deeper* are optional on a first pass: run their cells, then come back to them with the notes page.

**Studying on your own**

This notebook is written to be worked through alone as well as followed live. Allow about three hours for all of it. The ends of sections 2, 4 and 5 are good places to stop for a break.

| Sections | What you work through | About |
|---|---|---|
| 1 and 2 | the credit table, the split, new features, three encodings and the `Pipeline` | 40 min |
| 3 and 4 | blanks, outliers, two feature rankings and a measured removal | 45 min |
| 5 | two exercises on paper, then metrics, costs and the threshold | 60 min |
| 6 and 7 | L2, L1, a Keras network, cross-validation and the test set | 35 min |
| Recap | the Quick check | 10 min |

How to work a subsection:

- Read the text above the code cell and answer its **Predict first** question before you run anything. Write the answer down: a guess you have committed to is what makes the result stick.
- Run the cell. Compare what it prints with your prediction and with the bullets under it. The numbers quoted in the text come from our own run; with the seed set, counts should match exactly and scores should match or sit very close.
- Do the **Try changing** edit, see what moved, then put the value back and rerun the cell. Later cells rely on the original values.
- A **Common mistake** bullet names an error people often make at that step.
- **Work it by hand first** marks a paper exercise, at the start of section 5 and in 5.4. Do it before you read on.
- A line such as "Check your prediction" opens when you click it. Try first, then open it.
- Each section ends with **Before you move on**. If you cannot do one of its items, reread that subsection before going on.
- The Quick check at the end has its answers folded under it. Say each answer aloud before you open them: these are the questions an interviewer asks.
- If the runtime disconnects (Colab does this after a long idle spell), click the cell you were on and choose Runtime > Run before. Everything above it runs again. In our runs the whole notebook took about a minute, the two network fits in 6.3 about 15 seconds each.
- The [notes page for today](https://2494-gen-ai-cn.github.io/training/weeks/week-08/day-1-feature-engineering-and-model-evaluation.html) covers the same ideas in prose, with sources, worked arithmetic and more interview questions.

Run the next cell first. It holds the imports and the helpers we reuse all day, and it fixes the random seeds so your numbers match the ones quoted in the text.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import tensorflow as tf
from sklearn.base import clone
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, precision_recall_curve, roc_auc_score,
                             RocCurveDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)   # seeds Python, NumPy and TensorFlow together


def plot_history(history, metric="loss", title=""):
    """Draw the training and validation curves of one metric against the epoch, on one chart."""
    epochs = range(1, len(history.history[metric]) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history.history[metric], marker="o", markersize=4, linewidth=2, label=f"training {metric}")
    plt.plot(epochs, history.history["val_" + metric], marker="o", markersize=4, linewidth=2,
             label=f"validation {metric}")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


def total_cost(y_true, pred, cost_fn=5, cost_fp=1):
    """Total cost of 0/1 answers: cost_fn per defaulter approved (fn), cost_fp per good applicant refused (fp)."""
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return int(cost_fn * fn + cost_fp * fp)


def best_cost_threshold(y_true, proba, cost_fn=5, cost_fp=1):
    """Return the (threshold, cost) pair with the lowest total cost over the thresholds 0.05, 0.10, ..., 0.95."""
    pairs = [(round(float(t), 2), total_cost(y_true, (proba >= t).astype(int), cost_fn, cost_fp))
             for t in np.arange(0.05, 0.96, 0.05)]
    return min(pairs, key=lambda pair: pair[1])


def score_row(name, y_true, proba):
    """One results row at the default threshold 0.5: F1 and recall for "bad", and the total cost."""
    pred = (proba >= 0.5).astype(int)
    return {"model": name, "val_f1": f1_score(y_true, pred, zero_division=0),
            "val_recall": recall_score(y_true, pred), "val_cost": total_cost(y_true, pred)}


def keep_row(results, row):
    """Put a row in results, replacing an earlier row of the same model name, so a rerun adds no duplicate."""
    names = [r["model"] for r in results]
    if row["model"] in names:
        results[names.index(row["model"])] = row
    else:
        results.append(row)


print(f"Python {sys.version.split()[0]}, numpy {np.__version__}, pandas {pd.__version__}, "
      f"scikit-learn {sklearn.__version__}, TensorFlow {tf.__version__}, Keras {tf.keras.__version__}")
print("Setup complete")

## 1. Feature engineering

**Feature engineering** is building new input columns from the ones you have, and keeping each one only if it helps on validation rows. Before any feature is built, two things come first: deciding which columns may be used at all, and putting the test rows aside.

### 1.1 The credit table and the split

**credit-g** is the Statlog German credit data: 1,000 loan applicants from the 1990s, 20 attributes and no missing values, each applicant classed as a good or a bad credit risk. OpenML serves it, and `fetch_openml("credit-g", version=1, as_frame=True).frame` downloads it as one DataFrame. The exact `version` is passed because several versions of one dataset name can exist and differ.

In [ ]:
credit = fetch_openml("credit-g", version=1, as_frame=True).frame   # downloads the table from OpenML
print(credit.shape, credit.isna().sum().sum())                     # rows and columns, then blank cells
print(credit["class"].value_counts().to_dict())
credit.head(3)

- `(1000, 21) 0`: 21 columns (the 20 attributes and `class`) and not one blank cell.
- `class` is 700 good and 300 bad: a 70/30 table. Section 5 shows why that share matters for the choice of metric.
- Amounts are in Deutsche Mark, durations in months. The column `credit_amount` runs from a few hundred to over 18,000.

Three columns go before anything else is done. **A lender may not decide on them**, even if they would raise the score:

- `personal_status` mixes sex and marital status, `foreign_worker` records national origin, and `age` is age.
- The Canadian Human Rights Act lists national or ethnic origin, age, sex and marital status among its prohibited grounds of discrimination (section 3), and it covers services customarily available to the public (section 5).
- The US Equal Credit Opportunity Act's Regulation B counts national origin, sex, marital status and age as a prohibited basis. It allows age only in a statistically sound scoring system that gives an elderly applicant no negative value, so using age needs legal review.

The target is `y`, 1 for an applicant who defaulted. "Bad" is the **positive class**, so every precision, recall and F1 today is about the defaulters.

In [ ]:
PROTECTED = ["personal_status", "foreign_worker", "age"]   # sex and marital status, national origin, age
credit = credit.drop(columns=PROTECTED)                    # gone before any score is read
X = credit.drop(columns="class")
y = (credit["class"] == "bad").astype(int)                 # 1 = the applicant defaulted: "bad" is the positive class
print(f"{X.shape[1]} columns left; {y.sum()} defaulters of {len(y)}")

**The split comes first.** Any step that learns from rows, such as a median or a list of categories, must learn from the training rows only, so the rows are split before anything is learned. 20% (200 rows) go aside as the test set. A quarter of the remaining 800 become validation rows. `stratify` keeps the 30% share of defaulters in every part.

| Name | Rows | Used for |
|---|---|---|
| `X_tr`, `y_tr` | 600 | fitting every step and every model |
| `X_val`, `y_val` | 200 | comparing models, choosing the threshold |
| `X_test`, `y_test` | 200 | once, in section 7 |
| `X_trval`, `y_trval` | 800 | cross-validation in 6.4 (training and validation together) |

**Predict first:** the table is 30% defaulters and the split is stratified. How many defaulters will the 200 validation rows hold?

In [ ]:
X_trval, X_test, y_trval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)            # the test set, put aside first
X_tr, X_val, y_tr, y_val = train_test_split(
    X_trval, y_trval, test_size=0.25, stratify=y_trval, random_state=RANDOM_STATE)
print(len(X_tr), len(X_val), len(X_test), y_val.sum())   # training, validation, test rows; defaulters in validation

- `600 200 200 60`: 600 training rows, 200 validation, 200 test, and 60 defaulters in the validation rows, 30% of 200.
- The 60 defaulters are worth remembering: every validation score today rests on them, which is why section 6 asks how far one validation number can be trusted.
- **Common mistake:** dropping the protected columns after the split, or after looking at a score. They go before any number is read, because a column a lender may not use is ruled out whatever it does to the score. A feature built from one of them, or from a close stand-in for one, is ruled out with it.

**In an interview:** "I split before I learn anything from the data, so every fill, scale and category list comes from the training rows. And I leave out columns the decision may not rest on, such as sex, marital status, national origin and age in lending, before I look at a single score."

### 1.2 New features from old

Two features, each built from a row's own values:

- `monthly_payment = credit_amount / duration`: the loan amount divided by its months. Logistic regression only adds up its inputs, each multiplied by a weight, so it cannot divide one column by another by itself. A ratio that might matter has to be handed over as its own column.
- `duration_band`: `duration` cut into four bands with `pd.cut(..., bins=[0, 12, 24, 36, np.inf])`: up to 12 months, 13 to 24, 25 to 36, over 36. `pd.cut` makes **right-closed** bins by default, (0, 12], (12, 24] and so on, and a value outside every bin becomes a blank, so the last edge is `np.inf`.

Both read only their own row, so computing them on every part of the split leaks nothing. Neither is part of the model yet: section 4 judges both.

**Predict first:** a loan of exactly 12 months. Which band does it land in?

In [ ]:
def add_features(df):
    """Add two columns built from each row's own values."""
    return df.assign(
        monthly_payment=df["credit_amount"] / df["duration"],              # a ratio no single column holds
        duration_band=pd.cut(df["duration"], bins=[0, 12, 24, 36, np.inf],
                             labels=["<=12", "13-24", "25-36", ">36"]))   # right-closed: 12 lands in "<=12"

X_tr, X_val, X_test, X_trval = (add_features(d) for d in (X_tr, X_val, X_test, X_trval))
print(X_tr[["credit_amount", "duration", "monthly_payment", "duration_band"]].head(3).round(1))
print(X_tr["duration_band"].value_counts(sort=False).to_dict())

- The first training row: 4,811 over 30 months is about 160.4 a month, in the band "25-36".
- The 12-month loans (rows 754 and 209) land in "<=12", because each bin includes its right edge.
- Band counts on the 600 training rows: 222, 239, 83 and 56.
- **A feature from the future.** A column filled in after the loan was granted, such as a later missed payment, would score very well and be useless, because it carries the answer. Ask of every feature: does it exist at the moment the decision is made? Both of these do.
- **Common mistake:** counting a new column as a gain before it is measured. It is a candidate until 4.4 judges it on the validation rows.

**In an interview:** "Feature engineering builds new columns from existing ones, such as amount divided by duration, which a linear model cannot form by itself. I keep a feature only if the model does better with it on validation rows, and I check that it exists when the decision is made."

### 1.3 A log for a skewed column

*Going deeper.* The idea to keep: a log pulls in a long right tail and keeps the order of the values.

**Skew** measures how lopsided a column is: 0 is symmetric, and a positive number means a long right tail, a few values far above the rest. `np.log1p` takes the log of 1 plus the value. Inside a pipeline it is written `FunctionTransformer(np.log1p)`, which learns nothing from the rows. Section 3.3 uses it as one way to handle the largest loans.

In [ ]:
log_step = FunctionTransformer(np.log1p)                     # log(1 + x); learns nothing from the rows
amount = X_tr["credit_amount"]
amount_log = pd.Series(np.ravel(log_step.fit_transform(X_tr[["credit_amount"]])), index=X_tr.index)
print(f"credit_amount: {amount.min():,} to {amount.max():,}, skew {amount.skew():.2f}")
print(f"after the log: {amount_log.min():.2f} to {amount_log.max():.2f}, skew {amount_log.skew():.2f}")
print(f"largest loan before and after: row {amount.idxmax()} and row {amount_log.idxmax()}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(amount, bins=30, color="#4c72b0")
axes[0].set_title("credit_amount")
axes[1].hist(amount_log, bins=30, color="#55a868")
axes[1].set_title("log1p(credit_amount)")
plt.tight_layout()
plt.show()

- On the 600 training rows `credit_amount` runs from 276 to 14,782 with a skew of about 1.68. After the log it runs from 5.62 to 9.60 with a skew of about 0.12.
- The largest loan is the same row (374) before and after: a log is **monotonic**, so a larger amount always gives a larger log and the order of the values is kept.
- After the log the values are no longer amounts; only their order is. A decision tree, which splits one column at a threshold, has the same possible splits either way.

**Before you move on**, check that you can:

- say why `personal_status`, `foreign_worker` and `age` are dropped before any score is read;
- give the sizes of `X_tr`, `X_val` and `X_test` and say what each is for;
- build a ratio feature and a banded feature with `pd.cut`, and say which band a value on a bin edge lands in.

## 2. Encoding categorical data

A model reads numbers, and most columns of credit-g are categories such as `purpose` or `housing`. **Encoding** turns a category into numbers. Which encoding fits depends on two questions: do the values have a natural order, and how many values are there?

### 2.1 One-hot, again in one cell

**One-hot encoding** gives each value of a category its own 0-or-1 column. It suits a handful of values with no order. Last week's penguins build did it with `pd.get_dummies`; scikit-learn's `OneHotEncoder` does the same job and remembers the categories it was fitted on:

- `handle_unknown="ignore"` encodes a value the training rows never held as all zeros, instead of raising an error. Production will eventually send one.
- `sparse_output=False` returns an ordinary array. The default is a sparse matrix, which stores only the non-zero cells, and a Keras `Dense` layer (6.3) takes no sparse matrix.

**Predict first:** `purpose` has 10 values in the training rows. How many columns will one-hot make, and what will the row for an unseen purpose look like?

In [ ]:
one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)   # a dense array; every column kept
purpose_cols = one_hot.fit_transform(X_tr[["purpose"]])
print(purpose_cols.shape)                          # 600 rows, one column per purpose
print(list(one_hot.get_feature_names_out()[:4]))
print(purpose_cols[0])                             # the first training row: a single 1
print(one_hot.transform(pd.DataFrame({"purpose": ["drone"]})))   # a value the training rows never held

- `(600, 10)`: one column per purpose. Each name carries the column it came from, `purpose_business` and so on.
- The first training row has a single 1, in the column of its own purpose.
- "drone" was never seen, so its row is ten zeros.
- Every column is kept. Dropping one per category (`drop="first"`) breaks the symmetry of one-hot and can bias a penalised linear model such as today's.
- Try changing `handle_unknown="ignore"` to `handle_unknown="error"` and rerun: the last line raises `ValueError: Found unknown categories ['drone'] in column 0 during transform`. Put it back to `"ignore"`.
- **Common mistake:** leaving out `sparse_output=False`. The result is then a sparse matrix rather than an array, which prints differently and which Keras refuses in 6.3.

**In an interview:** "One-hot gives each category value its own 0-or-1 column, which suits a few values with no order. I set `handle_unknown="ignore"` so a value never seen in training becomes all zeros instead of an error."

### 2.2 Ordered categories

**Ordinal encoding** turns a category with a natural order into the integers 0, 1, 2 and so on, in that order, so one column carries it. A model reads integer codes as ordered, so this suits only a column whose values really are ordered.

`employment` is how long the applicant has held their present job: unemployed, under 1 year, 1 to under 4, 4 to under 7, 7 or more. That order is real. But the file stores its categories in some order of its own, and an encoder left to itself takes the stored order.

**Predict first:** if the stored order were alphabetical, which `employment` value would get the highest number?

In [ ]:
print(list(X_tr["employment"].cat.categories))       # the order the file stores

The stored order is alphabetical, so "unemployed" sorts last and would get the highest number, above seven years in one job. The real order has to be written out. `OrdinalEncoder` takes it as `categories=`, a **list of lists**: one inner list per encoded column.

In [ ]:
ORDERED = {"employment": ["unemployed", "<1", "1<=X<4", "4<=X<7", ">=7"]}   # the real order, written out
encoder = OrdinalEncoder(categories=[ORDERED["employment"]])    # a list of lists: one inner list per column
encoder.fit(X_tr[["employment"]])
print(encoder.transform(pd.DataFrame({"employment": ["unemployed", "1<=X<4", ">=7"]})).ravel())
print(list(X_tr["checking_status"].cat.categories))  # the fourth value is not a balance

- `[0. 2. 4.]`: unemployed is 0, 1 to 4 years is 2, 7 or more is 4. The written order holds.
- `checking_status` looks ordered (under 0, 0 to 200, 200 or more) until its fourth value, "no checking", which is not a balance at all. No single order fits all four, so it stays one-hot. `savings_status`, with "no known savings", has the same shape.

The next cell names the lists every pipeline today reads: the numbers, the ordered column and the one-hot columns. `duration_band` is in none of them yet; 4.4 tries it.

In [ ]:
ORDINAL = ["employment"]    # Try changing it: ORDINAL = [] and add "employment" to the end of ONE_HOT
ONE_HOT = ["purpose", "checking_status", "savings_status", "credit_history", "housing",
           "other_parties", "property_magnitude", "other_payment_plans", "job", "own_telephone"]
NUMERIC = ["duration", "credit_amount", "installment_commitment", "residence_since",
           "existing_credits", "num_dependents", "monthly_payment"]
print(f"{len(NUMERIC)} numeric, {len(ORDINAL)} ordinal and {len(ONE_HOT)} one-hot columns")

- 7 numeric columns (including `monthly_payment`), 1 ordinal column and 10 one-hot columns: the 18 columns the model reads.
- Try changing the first line to `ORDINAL = []` and add `"employment"` to the end of `ONE_HOT`. Do it once you have run 2.4: rerun this cell, then the two cells of 2.4. The encoded width grows from 51 to 55 (five one-hot columns replace one ordinal column), and in our run the baseline row moved from validation F1 0.425 and cost 209 to about 0.414 and 213. Put both lines back, then rerun this cell and the two cells of 2.4.
- **Common mistake:** `categories=ORDERED["employment"]`, a flat list without the outer brackets. The encoder expects one list per column and raises an error.

**In an interview:** "I ordinal-encode a category only when its values have a real order, and I write that order out, because the encoder otherwise takes the order the data stores. A category with a value that breaks the order, such as 'no checking account', stays one-hot."

### 2.3 When one-hot is wrong

**One-hot is the wrong choice for a column with hundreds of values**, such as a city or a product code. It makes hundreds of mostly-zero columns, which inflates the feature space, and production keeps sending values the training rows never held. Three ways out:

- `OneHotEncoder(min_frequency=..., max_categories=...)` groups rare values into one "infrequent" column: still one-hot, with fewer columns.
- **Frequency encoding** replaces each value by its share of the training rows: one column however many values there are. credit-g has no column with hundreds of values, so `purpose`, with 10, stands in for one here.
- **Target encoding** replaces each value by the average label of its rows. It suits many-valued columns, but it puts the label into the feature. scikit-learn's `TargetEncoder` (version 1.3 and later) cross-fits inside `fit_transform`, so each row's value comes from other rows' labels and the target does not leak.

A fourth way, named here only: a network can learn a short list of numbers for each value, a **learned embedding**. Vectors like these start tomorrow.

**Predict first:** frequency encoding maps each purpose to its share of the training rows. What does a purpose the training rows never held get?

In [ ]:
shares = X_tr["purpose"].value_counts(normalize=True)      # each value's share of the TRAINING rows
print(len(shares), shares[["new car", "retraining"]].round(3).to_dict())
X_val_purpose = X_val["purpose"].map(shares).astype(float)   # one column of numbers instead of ten
print(pd.DataFrame({"purpose": X_val["purpose"], "share": X_val_purpose.round(3)}).head(3))
unseen = pd.Series(["new car", "drone"]).map(shares).astype(float).fillna(0.0)   # "drone": no share, so 0
print(unseen.round(3).tolist())

- 10 purposes; "new car" is about 0.265 of the training rows and "retraining" about 0.007.
- Each validation row's purpose becomes one number, its share.
- An unseen value such as "drone" has no share, so `map` gives a blank, which the cell fills with 0.
- Two values with the same share get the same number, so frequency encoding cannot tell them apart. That is the price of one column.
- **Common mistake:** computing the shares on all 1,000 rows. The test rows would then shape a feature. The shares come from `X_tr` only and are applied to the other rows.

**In an interview:** "One-hot is wrong for a column with hundreds of values: too many sparse columns and a stream of unseen values. I would group rare values, use frequency encoding, or use target encoding cross-fitted so the label does not leak into the feature."

### 2.4 One object for every step

Two scikit-learn objects put every step together:

- A **`ColumnTransformer`** sends each list of columns to its own transformer and lays the outputs side by side. Columns in no list are dropped (`remainder="drop"`, the default). Here it is named `prep`: a `StandardScaler` on `NUMERIC`, the ordinal encoder on `ORDINAL`, one-hot on `ONE_HOT`.
- A **`Pipeline`** chains `prep` and the model, `LogisticRegression(max_iter=5000)` named `model`, into one object that is fitted, used and cross-validated as one.

`make_pipe` below builds that pipeline; its arguments swap one list or one step, which is how every later section builds its variants. Fitting it on `X_tr` learns every mean, scale and category list from those 600 rows; calling it on `X_val` only applies them.

**Predict first:** 7 numeric columns, 1 ordinal column, and one-hot on ten categories with between 2 and 10 values each. Will the encoded width be about 20, about 50 or about 100?

In [ ]:
def make_pipe(numeric=None, ordinal=None, one_hot=None, num_step=None, model=None):
    """Prepare the columns (prep) and fit a model (model), as one object; the lists default to 2.2's."""
    numeric = NUMERIC if numeric is None else numeric
    ordinal = ORDINAL if ordinal is None else ordinal
    one_hot = ONE_HOT if one_hot is None else one_hot
    prep = ColumnTransformer([
        ("num", StandardScaler() if num_step is None else num_step, numeric),
        ("ord", OrdinalEncoder(categories=[ORDERED[c] for c in ordinal]), ordinal),
        ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False), one_hot)])   # unlisted columns dropped
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=5000) if model is None else model)])

pipe = make_pipe().fit(X_tr, y_tr)              # every mean, scale and category list from the 600 training rows
prep = pipe.named_steps["prep"]
print(f"encoded width: {prep.transform(X_val).shape[1]} columns")
print(list(prep.get_feature_names_out()[[0, 7, 8]]))

- 51 columns: 7 scaled numbers, 1 ordinal column and 43 one-hot columns.
- `get_feature_names_out()` names each column with its transformer's prefix: `num__duration`, `ord__employment`, `oh__purpose_business`.

**The `results` table.** From here on, every model gets one row: validation F1 and recall for "bad", and `val_cost`, the total cost of its mistakes (section 5 explains the cost). **Every number in it is taken at the default threshold 0.5**, so that all rows compare; section 5 chooses a better threshold. `keep_row` replaces a row of the same name, so rerunning a cell adds no duplicate.

In [ ]:
results = []                       # one row per model; every number at the default threshold 0.5
pipes = {"baseline": pipe}          # each scikit-learn pipeline that has a row, by name
keep_row(results, score_row("baseline", y_val, pipe.predict_proba(X_val)[:, 1]))
pd.DataFrame(results).round(3)

- The baseline: validation F1 for "bad" 0.425, recall 0.40, cost 209. Every later change is compared with this row.
- Recall 0.40 means the model catches 24 of the 60 validation defaulters at threshold 0.5.
- **Common mistake:** `prep.fit_transform(X_val)`. That refits the preparation on the validation rows. Use the fitted pipeline (or `prep.transform`), which only applies what it learned from `X_tr`.

**In an interview:** "A `ColumnTransformer` sends each group of columns to its own encoder or scaler, and a `Pipeline` chains it with the model. Fitted on the training rows, it learns every preparation step from them alone, and cross-validation refits all of it inside each fold, so no held-out row leaks into the preparation."

**Before you move on**, check that you can:

- choose one-hot, ordinal or frequency encoding for a column from what its values mean and how many there are;
- write `OrdinalEncoder(categories=[[...]])` with the order spelled out, and say why the outer brackets are needed;
- say why one-hot is wrong for a column with hundreds of values, and name two alternatives;
- count the encoded width of a `ColumnTransformer` from its column lists.

## 3. Missing values and outliers

Blank cells and extreme values each need a decision, and each decision is measured on the validation rows like any other change.

### 3.1 Blanks, two ways

Three ways to handle a blank:

- **Drop the row.** Simple, but it throws away the rest of that row, data you may need. A last resort when few rows have blanks.
- **Fill it**, called **imputation**. `SimpleImputer` fills with the mean, the median (numbers only), the most frequent value or a constant, learned in `fit` from the training rows. The median is not pulled by extremes.
- **Fill it and flag it.** `add_indicator=True` adds a 0-or-1 column saying which cells were blank, so the model can use the fact that a value was missing.

**Predict first:** the column 1,000, blank, 3,000, 5,000 is filled with its median. What fills the blank, and what does the extra column hold with `add_indicator=True`?

In [ ]:
toy = pd.DataFrame({"credit_amount": [1000.0, np.nan, 3000.0, 5000.0]})
print(SimpleImputer(strategy="median").fit_transform(toy).ravel())
print(SimpleImputer(strategy="median", add_indicator=True).fit_transform(toy))   # second column: was it blank?

- The median of 1,000, 3,000 and 5,000 is 3,000, and it fills the blank.
- With the indicator, a second column reads 0, 1, 0, 0: a 1 on the row that was blank.

credit-g has no blanks, so the next cell blanks `credit_amount` in copies of the training and validation rows on purpose, two ways: once at random (20% of rows), and once mostly among defaulters (50% of defaulters, 5% of everyone else).

In [ ]:
def blank_amounts(df, labels, mostly_defaulters, seed):
    """Copy df with some credit_amount cells blanked: 20% of rows at random, or 50% of defaulters and 5% of the rest."""
    rng = np.random.default_rng(seed)
    labels = labels.to_numpy()
    share = np.where(labels == 1, 0.5, 0.05) if mostly_defaulters else np.full(len(df), 0.2)
    out = df.copy()
    out.loc[rng.random(len(out)) < share, "credit_amount"] = np.nan
    return out

copies = {"at random": (blank_amounts(X_tr, y_tr, False, 1), blank_amounts(X_val, y_val, False, 2)),
          "mostly defaulters": (blank_amounts(X_tr, y_tr, True, 1), blank_amounts(X_val, y_val, True, 2))}
for name, (tr, val) in copies.items():
    print(f"{name}: {tr['credit_amount'].isna().sum()} training and {val['credit_amount'].isna().sum()} "
          f"validation blanks; defaulters among the blank training rows {y_tr[tr['credit_amount'].isna()].mean():.2f}")

- At random: 117 training blanks, and the share of defaulters among them, 0.31, is close to the table's 0.30.
- Mostly among defaulters: 105 training blanks, 0.83 of them defaulters. Here a blank says something about the applicant.

Each copy now gets two pipelines: median fill, and median fill plus indicator. `monthly_payment` is left out of these four, because it still holds the amount the blank was meant to hide.

**Predict first:** on the copy with random blanks, will the indicator column raise the validation F1?

In [ ]:
rows = []
for name, (tr, val) in copies.items():
    for flag in (False, True):
        fill = Pipeline([("fill", SimpleImputer(strategy="median", add_indicator=flag)),   # median from tr only
                         ("scale", StandardScaler())])
        p = make_pipe(numeric=NUMERIC[:-1], num_step=fill).fit(tr, y_tr)   # no monthly_payment: it holds the amount
        rows.append({"blanks": name, "fill": "median + indicator" if flag else "median",
                     "val_f1": f1_score(y_val, p.predict(val))})
pd.DataFrame(rows).round(3)

- Random blanks: median 0.439, median plus indicator 0.421. A random blank carries no information, and the extra column only adds noise.
- Blanks mostly among defaulters: median 0.411, median plus indicator 0.593. The indicator column itself predicts default.
- That second case is called **missing not at random**: the blanks fall mostly on one kind of row, so the fact that a value is missing carries information. A blank that carries meaning earns its own column.
- **Common mistake:** blanking (or finding blanks in) only the validation rows. An indicator column appears only for a column that had blanks when the imputer was fitted, so with no blanks in `X_tr` the two pipelines are the same.

**In an interview:** "I fill blanks with a value learned from the training rows, usually the median for a skewed number, and I add an indicator column when the blanks may carry meaning. Whether the indicator helps is measured on validation rows; dropping rows is the last resort because it throws away the rest of each row."

### 3.2 Finding outliers

An **outlier** is a value far from most of a column's values. The usual yardstick is the **interquartile range**:

- **Q1** is the value a quarter of the way up the sorted column, **Q3** three quarters of the way up. The IQR is Q3 - Q1, the width of the middle half.
- **The 1.5 IQR rule:** a point below Q1 - 1.5 IQR or above Q3 + 1.5 IQR (the inner fences) is a mild outlier; beyond 3 IQR it is an extreme one.

A box plot draws exactly this: the box is the middle half, and the whiskers stop at the last points inside the fences.

In [ ]:
plt.figure(figsize=(9, 2.5))
sns.boxplot(x=X_tr["credit_amount"], color="#8ab6d6")    # whiskers stop at the last points inside 1.5 IQR
plt.title("credit_amount on the 600 training rows")
plt.xlabel("credit amount (Deutsche Mark)")
plt.show()

The box sits low and the right whisker is long, with a scatter of points beyond it: a few large loans far above the rest. The next cell computes the fences from the training rows.

**Predict first:** on the training rows, Q1 is about 1,345 and Q3 about 4,042. Where is the upper fence?

In [ ]:
q1, q3 = X_tr["credit_amount"].quantile([0.25, 0.75])   # from the training rows only
lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
print(f"Q1 {q1:,.0f}, Q3 {q3:,.0f}, IQR {q3 - q1:,.0f}, fences {lower:,.0f} and {upper:,.0f}")
print(f"above the upper fence: {(X_tr['credit_amount'] > upper).sum()}, below the lower: "
      f"{(X_tr['credit_amount'] < lower).sum()}")
X_tr[X_tr["credit_amount"] > upper][["credit_amount", "duration", "purpose"]].sort_values("credit_amount").tail(3)

- IQR 2,697, so 1.5 IQR is about 4,046 and the upper fence about 8,088. The lower fence, about -2,701, is below zero.
- 41 training loans sit above the upper fence and none below it, since no amount is negative.
- The three largest are 14,318 to 14,782 over 36 to 60 months: large, but possible. **Error or real?** An impossible value (a negative amount, a 900-month loan) is an error to fix. A large possible one is data, and 3.3 shows three ways to keep its effect in check.
- **Common mistake:** computing the fences on all 1,000 rows. Like every learned step, they come from `X_tr` only.

**In an interview:** "I flag a value as an outlier when it is more than 1.5 interquartile ranges beyond the middle half, with the quartiles from the training rows. Then I ask whether it is an error to fix or a real extreme value, and keep a real one."

### 3.3 Keep, clip or log

Three ways to handle the large loans, each its own pipeline scored on the same validation rows:

- **Keep** the values and use a scaler that ignores extremes. `RobustScaler` subtracts the median and divides by the IQR, because outliers pull the mean and variance that `StandardScaler` uses.
- **Clip** every amount at the fence from `X_tr`: a loan above 8,088 counts as 8,088. Every row stays.
- **Log** the amount with `FunctionTransformer(np.log1p)`, as in 1.3.

The choice matters for linear models and networks, whose weighted sums feel the size of a value. A decision tree splits one column at a threshold, so it hardly notices.

**Predict first:** of keep, clip and log, which one does not keep the order of every amount?

In [ ]:
def amount_step(func, **kw_args):
    """Apply func to credit_amount only, pass the other numbers through, then scale them all."""
    first = ColumnTransformer([("amount", FunctionTransformer(func, kw_args=kw_args), ["credit_amount"])],
                              remainder="passthrough")
    return Pipeline([("amount", first), ("scale", StandardScaler())])

clip_low, clip_high = lower, upper   # Try changing it: clip_low, clip_high = X_tr["credit_amount"].quantile([0.05, 0.95])
print(f"clipping at {clip_low:,.0f} and {clip_high:,.0f}: {(X_tr['credit_amount'] < clip_low).sum()} training rows "
      f"raised, {(X_tr['credit_amount'] > clip_high).sum()} lowered")
runs = {"keep (RobustScaler)": make_pipe(num_step=RobustScaler()),
        "clip": make_pipe(num_step=amount_step(np.clip, a_min=clip_low, a_max=clip_high)),
        "log": make_pipe(num_step=amount_step(np.log1p))}
for name, p in runs.items():
    pipes[name] = p.fit(X_tr, y_tr)
    keep_row(results, score_row(name, y_val, p.predict_proba(X_val)[:, 1]))
pd.DataFrame(results).round(3)

- Clip loses the order: the 41 largest amounts all become 8,088, so their order among themselves is gone. The log keeps it.
- In our run keep (`RobustScaler`) scored exactly like the baseline, F1 0.425 and cost 209, while clip and log both reached about 0.439 and cost 204. One more defaulter caught out of 60: a small difference on 200 rows.
- Try changing the `clip_low, clip_high` line to `clip_low, clip_high = X_tr["credit_amount"].quantile([0.05, 0.95])` and rerun: the bounds become about 707 and 8,949, so 30 training rows are raised at the bottom and 30 lowered at the top (5% of 600 each), instead of 41 at the top only. In our run the clip row stayed at about 0.439 and 204. Put the line back to `clip_low, clip_high = lower, upper` and rerun the cell.
- **Common mistake:** taking the clip bounds from the validation rows. The bounds come from `X_tr` and are applied to both.

**In an interview:** "For a linear model or a network I keep a real extreme value but limit its pull: a robust scaler, clipping at a fence from the training rows, or a log. I pick by validation score; for a tree model it hardly matters, because a tree only splits on order."

### 3.4 IsolationForest flags unusual rows

*Going deeper.* The idea to keep: a row can be unusual as a whole even when none of its values is beyond a fence.

`IsolationForest` isolates rows with random splits. An unusual row takes fewer splits to separate from the rest, a shorter path. `predict` returns -1 for a row it calls unusual and 1 for the others. It is fitted on the encoded training rows, with its default `contamination="auto"`.

In [ ]:
X_tr_enc = prep.transform(X_tr)                     # 2.4's prep: the 51 encoded columns
iso = IsolationForest(random_state=RANDOM_STATE).fit(X_tr_enc)   # contamination="auto", the default
flags = iso.predict(X_tr_enc)                       # -1 unusual, 1 usual
unusual = flags == -1
print(f"{unusual.sum()} of {len(flags)} training rows flagged as unusual")
print(f"of those, {(X_tr['credit_amount'][unusual] > upper).sum()} are above 3.2's fence")
print(f"defaulters: {y_tr[unusual].mean():.2f} among flagged rows, {y_tr[~unusual].mean():.2f} among the rest")
X_tr.loc[unusual, ["credit_amount", "duration", "purpose", "housing", "job"]].head(4)

- 121 of 600 training rows are flagged, and only 21 of them are among the 41 large loans of 3.2: most flagged rows are unusual for their combination of values.
- Defaulters make up about 0.41 of the flagged rows against 0.27 of the rest. Even so, -1 means unusual, not defaulter.
- As in 3.2, a flag starts a check (error or real?), not a deletion.

**Before you move on**, check that you can:

- choose between dropping, filling and filling with an indicator, and say when the indicator earns its column;
- compute the 1.5 IQR fences from Q1 and Q3 and say which rows they flag;
- name three ways to handle a real extreme value and say which kinds of model care.

## 4. Feature selection techniques

### 4.1 Why fewer features

Four reasons to drop columns:

- **Noise.** A column with no signal gives the model something to fit by chance.
- **Cost.** Every column has to be collected, checked and kept up to date in production.
- **Explanation.** A model on ten columns is easier to explain to the people it decides about.
- **Columns a lender may not use.** `personal_status`, `foreign_worker` and `age` were dropped in 1.1 and are never offered to a ranking. Using age needs legal review.

Selection learns from data, so it follows the split-first rule. Three families of method:

- A **filter** scores each column against the target with no model (mutual information, 4.2).
- A **model-based ranking** asks a fitted model which columns it needs (permutation importance, 4.3).
- A **penalty** built into training sets some weights to exactly zero (L1, 6.2).

**Predict first:** you rank the columns on all 1,000 rows, keep the best ones, and only then split. Will the validation score be honest?

<details>
<summary>Check your prediction</summary>

No. The ranking has already seen the rows that later score the model, so it favours columns that happen to fit them. scikit-learn's guide shows the size of the effect: on pure noise, where 0.5 is the honest score, selecting before the split scored about 0.76.

</details>

**In an interview:** "I drop features for noise, cost, explanation, and because some columns may not be used at all. I rank them with a filter or with the fitted model, always on rows the final score does not depend on, and I measure the removal on validation rows."

### 4.2 A filter: mutual information

**Mutual information** measures how much knowing a column tells you about the target. `mutual_info_classif` returns one non-negative score per column, zero only when the column and the target are independent. It is estimated from nearest-neighbour distances, which carry some randomness that `random_state` fixes. It scores the 51 encoded training columns one at a time, with no model.

**Predict first:** a column with nothing to do with default. What score should it get?

In [ ]:
mi = mutual_info_classif(prep.transform(X_tr), y_tr, random_state=RANDOM_STATE)   # the training rows only
ranking = pd.Series(mi, index=prep.get_feature_names_out()).sort_values(ascending=False)
print(ranking.head(6).round(3).to_string())
print(f"columns scored exactly 0: {(ranking == 0).sum()} of {len(ranking)}")

- The top six: `checking_status` "no checking" (about 0.048), `credit_amount` (0.036), `duration` (0.034), `other_payment_plans` "none" (0.031), `checking_status` "<0" (0.029) and `installment_commitment` (0.029).
- 21 of the 51 columns score exactly 0: the estimate found no link at all for them in these 600 rows.
- The scores are rough. With its default `discrete_features="auto"` the function treats every column of a dense array as continuous, though the one-hot columns are really 0 or 1.
- A filter scores each column alone, so it cannot see a column that helps only together with another.
- **Common mistake:** running the filter on `X_val` or on all the rows. It learns from the training rows only.

**In an interview:** "A filter method such as mutual information scores each column against the target without a model. It is fast and model-free, but it sees one column at a time and misses columns that only help in combination."

### 4.3 A model-based ranking

**Permutation importance** shuffles one column, which breaks its link to the target, and records how much the score drops. `n_repeats=10` averages ten shuffles. It runs on the validation rows, so it shows which columns the model needs on data it has not seen; on training rows, a column the model memorised would look important too. A fitted pipeline takes the original columns, so the ranking is per original column (18 of them), not per encoded column.

**Predict first:** will `monthly_payment`, the feature built in 1.2, rank near the top?

In [ ]:
FEATURES = NUMERIC + ORDINAL + ONE_HOT              # the 18 original columns the pipeline reads
imp = permutation_importance(pipe, X_val[FEATURES], y_val, scoring="f1", n_repeats=10,
                             random_state=RANDOM_STATE)  # on the validation rows
by_column = pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False)
print("top 5:", by_column.head(5).round(3).to_dict())
print("below 0:", by_column[by_column < 0].round(3).to_dict())

- The top five: `savings_status` (about 0.040 F1), `duration` (0.038), `property_magnitude` (0.035), `employment` (0.034) and `checking_status` (0.030).
- Six columns are below zero, `monthly_payment` among them at about -0.012. A **negative importance** means the score went up when the column was scrambled: what the model learned from it does not hold on new rows.
- The two rankings disagree in places: the filter scores encoded columns alone, permutation scores original columns inside the fitted model.
- **Common mistake:** reading a low importance as "useless". Two columns that carry the same information, such as `credit_amount` and `monthly_payment`, share their importance, and both can look weak.

**In an interview:** "Permutation importance shuffles one column on held-out rows and measures how much the score drops. I compute it on validation rows, because on training rows a memorised column looks important, and I remember that correlated columns split their importance."

### 4.4 Does dropping help?

Removing columns is a change like any other: refit without them and compare on the same validation rows. Three rows go into `results`: only the ten columns with the highest permutation importance, everything except `monthly_payment`, and everything plus 1.2's `duration_band` (one-hot, four more columns).

**Predict first:** will dropping `monthly_payment` lower the validation F1?

In [ ]:
top10 = by_column.index[:10].tolist()
print("top 10:", top10)
runs = {"top 10 columns": make_pipe(numeric=[c for c in NUMERIC if c in top10],
                                    ordinal=[c for c in ORDINAL if c in top10],
                                    one_hot=[c for c in ONE_HOT if c in top10]),
        "no monthly_payment": make_pipe(numeric=NUMERIC[:-1]),
        "with duration_band": make_pipe(one_hot=ONE_HOT + ["duration_band"])}   # 1.2's band, one-hot
for name, p in runs.items():
    pipes[name] = p.fit(X_tr, y_tr)
    keep_row(results, score_row(name, y_val, p.predict_proba(X_val)[:, 1]))
pd.DataFrame(results).round(3)

- No monthly_payment: exactly the baseline, 0.425 and cost 209. The feature from 1.2 did not earn its place.
- With duration_band: exactly the baseline too. Neither new feature helped this model on these rows, and the simpler model wins a tie.
- Top 10 columns: F1 about 0.438 and cost 207, slightly better than the baseline with eight fewer columns. That row is flattered: the ranking that chose the ten came from these same validation rows.
- **Common mistake:** trusting the top-10 row as a fair score. The same rows chose and scored it; cross-validation (6.4) is the fairer test.

**In an interview:** "I judge a removal the way I judge any change: refit without the columns and compare on the same validation rows, or better with cross-validation, because a selection scored on the rows that chose it is optimistic."

**Before you move on**, check that you can:

- name a filter method and a model-based method and say what each one looks at;
- say why permutation importance runs on validation rows, and what a negative importance means;
- say whether `monthly_payment` earned its place, and from which row of `results`.

## 5. Choosing a metric

Last week's confusion matrix, precision, recall and F1 carry on here. This section starts from what they leave open: which number to report, how to average it, and where to put the threshold.

**Work it by hand first.** Before you run anything, answer three lines for each case: who owns the decision, which mistake costs more, and which number you would report.

```
Case 1. Loan approval. A model reads a loan application and answers "will default" (refuse) or "will repay" (approve).
Case 2. Spam filter. A model reads each incoming email and answers "spam" (moved to the spam folder) or "not spam" (left in the inbox).
Case 3. Cancer screening. A model reads a screening result and answers "follow up" (a further test) or "all clear".
```

<details>
<summary>Check your answers</summary>

| Case | Who owns the decision | Costlier mistake | Report |
|---|---|---|---|
| Loan approval | the lender's credit team | approving a defaulter: credit-g's own costs make it five times worse than refusing a good applicant | recall on the defaulters and the total cost |
| Spam filter | the product owner of the email service | either, with a reason; if a lost real email is worse, it is the false positive (with "spam" positive) | precision on "spam" if real emails must not be lost |
| Cancer screening | the clinic | a missed cancer costs far more than an extra follow-up test | recall on the cancer cases, with the threshold lowered to favour it |

The pattern: the people who own the decision say which mistake costs more, and the metric and the threshold follow from that.

Usual wrong answers:

- "Accuracy" for any of the three. On unbalanced classes accuracy hides the mistake that matters (5.1 shows it).
- "The data scientist decides which mistake is worse." The data scientist asks and turns the answer into a metric; the owner of the decision answers.

</details>

### 5.1 Accuracy on a 70/30 table

**Accuracy** is the share of answers that are right. 70% of the validation applicants are good, so a model that answers "good" for everyone scores 70% without reading a single column. **Balanced accuracy** is the average of each class's recall, so it gives such a model 0.5.

**Predict first:** which scores higher on accuracy, "always good" or the baseline pipeline?

In [ ]:
pred = pipe.predict(X_val)                         # the baseline pipeline at threshold 0.5
always_good = np.zeros(len(y_val), dtype=int)       # "good" for every applicant
for name, guess in [("always good", always_good), ("baseline pipeline", pred)]:
    print(f"{name}: accuracy {accuracy_score(y_val, guess):.2f}, recall {recall_score(y_val, guess):.2f}, "
          f"balanced accuracy {balanced_accuracy_score(y_val, guess):.2f}, defaulters caught {guess[y_val == 1].sum()}")

- "Always good": accuracy 0.70, recall 0.00, balanced accuracy 0.50, and no defaulter caught.
- The baseline pipeline: accuracy 0.68, recall 0.40, balanced accuracy 0.60, and 24 of the 60 defaulters caught.
- The model that catches no defaulter wins on accuracy. On a 70/30 table accuracy hides the defaulters.
- **Common mistake:** dismissing the pipeline because its accuracy is lower. Its recall of 0.40 is what the lender pays for.

**In an interview:** "On an unbalanced table accuracy can be beaten by a model that always answers the majority class, so I report the scores of the class that matters, or balanced accuracy, or better the cost of the mistakes."

### 5.2 Precision, recall and F1 for the defaulters, and their averages

`classification_report` prints precision, recall, F1 and **support** (the number of true rows) for each class, then accuracy and two averages:

- The **macro average** is the plain mean of the per-class scores: every class counts the same, so a rare class with a low score drags it down.
- The **weighted average** weights each class by its support, so the larger class pulls it towards its own score.

**Predict first:** the good class has 140 rows and an F1 near 0.77; the bad class has 60 rows and an F1 near 0.42. Will the weighted F1 be closer to 0.77 or to 0.42?

In [ ]:
print(classification_report(y_val, pred, target_names=["good", "bad"], digits=3))

Read it row by row:

- The "bad" row: precision 0.453 (of the 53 applicants called bad, 24 really default), recall 0.400 (24 of the 60 defaulters caught), F1 0.425. These are the defaulter scores.
- The "good" row: F1 0.774, on 140 rows.
- No micro average row: for a problem with one label per row and every class included, micro averaging equals accuracy, which is already printed.

In [ ]:
per_class = f1_score(y_val, pred, average=None)          # one F1 per class: good, bad
print(f"F1 good {per_class[0]:.3f}, bad {per_class[1]:.3f}")
print(f"macro    = ({per_class[0]:.3f} + {per_class[1]:.3f}) / 2 = {f1_score(y_val, pred, average='macro'):.3f}")
print(f"weighted = (140 * {per_class[0]:.3f} + 60 * {per_class[1]:.3f}) / 200 = "
      f"{f1_score(y_val, pred, average='weighted'):.3f}")

- Macro F1 = (0.774 + 0.425) / 2, about 0.599. Weighted F1 = (140 × 0.774 + 60 × 0.425) / 200, about 0.669, pulled up by the larger good class.
- Macro F1 is the mean of the per-class F1s, not an F1 worked out from mean precision and mean recall.
- **F1 is a harmonic mean** of precision and recall: F1 = 2 tp / (2 tp + fp + fn). For the defaulters, 2 × 24 / (48 + 29 + 36) = 48 / 113, about 0.425. A harmonic mean is pulled towards the smaller number: precision 1.0 with recall 0.1 averages 0.55 but has an F1 of about 0.18.
- A model that never predicts "bad" has precision 0 / 0. scikit-learn reports 0 with a warning (`zero_division="warn"`, the default); 6.1's most penalised model is one of these.
- **Common mistake:** reading the weighted 0.669 as the defaulter score. The defaulter F1 is 0.425, on the "bad" row.

**In an interview:** "Macro averaging takes the plain mean of the per-class scores, so a weak rare class shows; weighted averaging weights by support and can hide it. When one class matters, I report that class's precision, recall and F1 rather than an average."

### 5.3 The confusion matrix as costs

In `confusion_matrix` the **rows are the true labels and the columns the predicted ones**. For two classes, `.ravel()` flattens it into `tn, fp, fn, tp`. With "bad" as 1:

- a **false negative** (fn) is a defaulter approved;
- a **false positive** (fp) is a good applicant refused.

credit-g comes with its own **cost matrix**: 0 for a correct answer, 5 for a defaulter approved, 1 for a good applicant refused. So the total cost of a model is `5 * fn + 1 * fp`, the `val_cost` column of `results`.

**Predict first:** with costs of 5 and 1, roughly what share of the baseline's total cost will come from approved defaulters?

In [ ]:
cm = confusion_matrix(y_val, pred)                 # rows: the truth (good, bad); columns: the prediction
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"defaulters approved {fn} x 5 = {5 * fn}, good applicants refused {fp} x 1 = {fp}, total cost {5 * fn + fp}")
print(f"'always good' approves all {y_val.sum()} defaulters: cost {5 * y_val.sum()}")

- `[[111, 29], [36, 24]]`. Top left, 111 good applicants approved: cost 0. Top right, 29 good applicants refused: 1 each. Bottom left, 36 defaulters approved: 5 each, so 180. Bottom right, 24 defaulters refused: cost 0.
- Total 180 + 29 = 209, against 300 for "always good". Most of the cost, 180 of 209, sits in one cell: the approved defaulters.

Now the whole `results` table, cheapest first:

In [ ]:
pd.DataFrame(results).round(3).sort_values("val_cost")    # every row at threshold 0.5

- Clip and log lead at 204, then the top-10 row at 207, then four rows at 209.
- A lower F1 can come with a lower cost: F1 counts both mistakes alike, while the cost weighs a missed defaulter five times. That is why `results` carries both.
- **Common mistake:** swapping rows and columns. Rows are the truth, columns the prediction, so the bottom-left cell is the defaulters approved.

**In an interview:** "I read a confusion matrix with the truth in the rows and the prediction in the columns, then multiply each cell by what that mistake costs the business. The total cost compares models in the currency the decision owner cares about."

### 5.4 The decision threshold

`predict_proba(X_val)[:, 1]` gives each applicant's probability of default, and `(proba >= t).astype(int)` turns it into a decision at **threshold** t. `predict` uses 0.5, a cut-off scikit-learn's own guide calls "most certainly not ideal for most use cases". The threshold should come from the costs.

**Work it by hand first.** An applicant's chance of default is p. Approving costs 5 if they default, so p × 5 on average. Refusing costs 1 if they would have repaid, so (1 - p) × 1 on average. Work out, on paper:

1. the value of p above which refusing is cheaper;
2. whether to approve an applicant with p = 0.25;
3. the value from step 1 if an approved defaulter cost 10 instead of 5.

<details>
<summary>Check your answers</summary>

1. Refuse when 5p > 1 - p, so 6p > 1, so p > 1 / (1 + 5) = 1/6, about 0.17. In general (Elkan's rule): refuse when p is above the cost of refusing a good applicant divided by the sum of the two costs.
2. Refuse. Approving costs 0.25 × 5 = 1.25 on average, refusing 0.75 × 1 = 0.75.
3. 1 / (1 + 10) = 1/11, about 0.09. A costlier missed defaulter means refusing sooner.

Usual wrong answers:

- 1/5 = 0.20: the cost ratio, not the threshold. The threshold divides by the sum of both costs, 1 + 5.
- "Approve at 0.25, because it is below 0.5." That is the 0.5 habit; with a 5-to-1 cost the line sits near 0.17.

</details>

The cell sweeps the pipeline with the lowest `val_cost` in `results` so far, and prints its name. `best_cost_threshold` (from the setup cell) tries the thresholds 0.05, 0.10 and so on up to 0.95 and returns the cheapest.

**Predict first:** from the 5-and-1 costs, near which threshold should the total cost be lowest?

In [ ]:
cheapest = min((r for r in results if r["model"] in pipes), key=lambda r: r["val_cost"])["model"]
proba = pipes[cheapest].predict_proba(X_val)[:, 1]   # each validation applicant's chance of default
print(f"sweeping the lowest-cost pipeline so far: {cheapest}")
COST_FN, COST_FP = 5, 1          # Try changing it: COST_FN, COST_FP = 10, 1
for t in (0.10, 0.15, 0.20, 0.50):
    tn, fp, fn, tp = confusion_matrix(y_val, (proba >= t).astype(int)).ravel()
    print(f"threshold {t:.2f}: defaulters approved {fn}, good refused {fp}, cost {COST_FN * fn + COST_FP * fp}")
print(f"lowest-cost (threshold, cost): {best_cost_threshold(y_val, proba, COST_FN, COST_FP)}")
print(f"the cost rule: {COST_FP} / ({COST_FP} + {COST_FN}) = {COST_FP / (COST_FP + COST_FN):.3f}")

- The lowest-cost pipeline so far is `clip` (tied with `log` at 204; `min` keeps the first).
- At 0.50: 35 defaulters approved, 29 good applicants refused, cost 204. At 0.20: 12 and 62, cost 122. At 0.15: 6 and 75, cost 105. At 0.10: 2 and 89, cost 99.
- The cheapest threshold is 0.10, at a cost of 99, half the cost at 0.5. Lowering the threshold refuses more people: far fewer defaulters slip through, at the price of more good applicants refused, and under these costs that trade is worth it.
- The rule says about 0.17. The grid steps by 0.05, so it has no 0.17, and the sweep's lowest point lies a little below it. The rule assumes the probabilities are **calibrated** (a 0.2 really is a 20% chance), and 200 rows with 60 defaulters give a noisy answer, so near the rule is what to expect.

The same sweep, drawn: precision, recall and total cost against the threshold, with the cost rule as a dashed line.

In [ ]:
grid = np.arange(0.05, 0.96, 0.05)
answers = [(proba >= t).astype(int) for t in grid]
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(grid, [precision_score(y_val, a, zero_division=0) for a in answers], marker="o", label="precision (bad)")
ax.plot(grid, [recall_score(y_val, a) for a in answers], marker="o", label="recall (bad)")
ax.set_xlabel("threshold")
ax.set_ylabel("precision and recall")
ax.grid(alpha=0.3)
cost_ax = ax.twinx()                                   # a second y axis for the cost
cost_ax.plot(grid, [total_cost(y_val, a, COST_FN, COST_FP) for a in answers], color="black", marker="s",
             label="total cost")
cost_ax.axvline(COST_FP / (COST_FP + COST_FN), color="grey", linestyle="--", label="cost rule")
cost_ax.set_ylabel("total cost on the 200 validation rows")
fig.legend(loc="upper center", ncol=4)
plt.show()

- Recall rises as the threshold falls, because more applicants are refused; precision mostly falls, because more of those refused are good applicants. At 0.95 nobody is refused, so precision shows as 0.
- The cost curve has its low point at 0.10, just left of the dashed rule, and is fairly flat there: 0.05, 0.10 and 0.15 cost 116, 99 and 105, against 204 at 0.5 and 300 at 0.95.
- Try changing the line `COST_FN, COST_FP = 5, 1` to `COST_FN, COST_FP = 10, 1` and rerun the cell above and this one: the rule moves to 1/11, about 0.091, and every total that includes an approved defaulter goes up (at 0.50 from 204 to 379). In our run the cheapest threshold stayed at 0.10, now at a cost of 109: the grid has no point between 0.05 and 0.10, and 0.05 still costs more. Put it back to `5, 1` and rerun both cells.
- **Common mistake:** choosing the threshold on `X_test` or on the training rows, or reusing it for another model. It is tuned on the validation rows for the model it is used with; the test set waits for section 7.

**In an interview:** "I choose the threshold from the cost of each mistake. With no cost for a correct answer, the cut-off is the false-positive cost over the sum of the two costs, 1/6 for credit-g's 5 to 1. Then I check it with a sweep on validation probabilities of the model I keep, never on test or training rows."

### 5.5 ROC and precision-recall curves

*Going deeper.* The idea to keep: AUC measures how well a model ranks positives above negatives over every threshold; 0.5 is random.

One confusion matrix describes one threshold. The **ROC curve** describes all of them: it plots the true positive rate (recall) against the false positive rate, fp / (fp + tn), as the threshold moves. The **AUC** is the area under it: 1.0 for a perfect ranking, 0.5 for a random one. `roc_auc_score` takes the probabilities, not the 0-or-1 answers. The **precision-recall curve** plots the same sweep as precision against recall, which tells more when positives are few, because precision counts the false alarms directly.

In [ ]:
auc = roc_auc_score(y_val, proba)                  # the probabilities, not the 0/1 answers
print(f"AUC {auc:.3f}, Gini {2 * auc - 1:.3f}")
precision, recall, thresholds = precision_recall_curve(y_val, proba)
print(len(precision), len(thresholds), precision[-1], recall[-1])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
RocCurveDisplay.from_predictions(y_val, proba, ax=axes[0], plot_chance_level=True)
axes[0].set_title("ROC curve")
axes[1].plot(recall, precision)
axes[1].axhline(y_val.mean(), color="grey", linestyle="--", label="share of defaulters")
axes[1].set_xlabel("recall (bad)")
axes[1].set_ylabel("precision (bad)")
axes[1].set_title("Precision-recall curve")
axes[1].legend()
plt.tight_layout()
plt.show()

- AUC about 0.717 on the validation rows: well above random, far from perfect. Gini, 2 × AUC - 1, is about 0.435.
- `precision_recall_curve` returned 201 precision values and 200 thresholds: the curve ends at one extra point, precision 1.0 and recall 0.0, with no threshold of its own.
- The precision-recall chart's dashed line is the share of defaulters, 0.30: what precision a random ranking would give.
- Neither curve says which threshold to use; the costs do (5.4).

**Before you move on**, check that you can:

- say why "always good" scores 0.70 accuracy and why that makes accuracy the wrong number here;
- compute macro and weighted F1 from the two per-class F1s and their supports;
- turn a confusion matrix into a total cost with 5 per defaulter approved and 1 per good applicant refused;
- work out the cost-minimising threshold from two costs, and say on which rows the sweep runs.

## 6. Avoiding overfitting: regularisation and cross-validation

Last week fought overfitting with dropout and early stopping. **Regularisation** is a third way: a penalty on large weights, added to the training loss, so the model stays simpler and overfits less.

### 6.1 L2 in scikit-learn

**L2** adds a cost proportional to the squared weights. It shrinks every weight towards zero without setting any to exactly zero. It is week 6's ridge, and `LogisticRegression` uses it by default.

**`C`** sets its strength the other way round: `C` is the inverse of the strength, default 1.0, so a **smaller `C` is a stronger penalty**. The sweep refits the pipeline at six values of `C` and records training and validation F1, and the largest weight.

**Predict first:** at which `C` will the model predict no defaulter at all?

In [ ]:
sweep = []
for C in [0.001, 0.01, 0.1, 1, 10, 100]:            # smaller C = stronger penalty
    m = make_pipe(model=LogisticRegression(C=C, max_iter=5000)).fit(X_tr, y_tr)
    sweep.append({"C": C, "train_f1": f1_score(y_tr, m.predict(X_tr), zero_division=0),
                  "val_f1": f1_score(y_val, m.predict(X_val), zero_division=0),   # 0 when no "bad" is predicted
                  "largest_weight": np.abs(m.named_steps["model"].coef_).max()})
sweep = pd.DataFrame(sweep)
print(sweep.round(3).to_string(index=False))

The same sweep, drawn on a log axis:

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(sweep["C"], sweep["train_f1"], marker="o", linewidth=2, label="training F1")
plt.plot(sweep["C"], sweep["val_f1"], marker="o", linewidth=2, label="validation F1")
plt.xscale("log")
plt.xlabel("C (log scale; left = stronger penalty)")
plt.ylabel("F1 for 'bad'")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

- At `C=0.001` the penalty squeezes every weight close to zero (the largest is about 0.05), so no probability reaches 0.5 and the model predicts no defaulter: F1 0 on both sets. That is **underfitting**.
- `C=0.01`: training F1 about 0.22, validation 0.06. `C=0.1`: 0.58 and 0.41. From `C=1` up, training F1 sits near 0.62 to 0.63 and validation near 0.43 to 0.44.
- The gap of about 0.2 between training and validation is the overfitting, and on 600 rows a weaker penalty does not close it. The largest weight grows as the penalty weakens.
- **Common mistake:** reading `C` backwards. A larger `C` is a weaker penalty.

**In an interview:** "L2 regularisation adds the squared weights to the loss, which shrinks them towards zero. In scikit-learn's `LogisticRegression`, `C` is the inverse of the strength, so a smaller `C` regularises more, and too small a `C` underfits."

### 6.2 L1 sets weights to zero

**L1** adds a cost proportional to the absolute size of each weight. It pushes many weights to exactly zero, so it selects features as it trains: week 6's lasso. A **solver** is the algorithm that searches for the weights. The default, `lbfgs`, supports only L2 or no penalty and raises an error for L1, so the cell names `solver="liblinear"`.

**Predict first:** at the same `C=0.1`, will L1 or L2 set more of the 51 weights to exactly zero?

In [ ]:
l1 = make_pipe(model=LogisticRegression(penalty="l1", solver="liblinear", C=0.1)).fit(X_tr, y_tr)
l2 = make_pipe(model=LogisticRegression(C=0.1, max_iter=5000)).fit(X_tr, y_tr)
for name, m in [("L1", l1), ("L2", l2)]:
    w = m.named_steps["model"].coef_.ravel()           # one weight per encoded column
    print(f"{name} at C=0.1: {(w == 0).sum()} of {w.size} weights are exactly zero, "
          f"validation F1 {f1_score(y_val, m.predict(X_val)):.3f}")
kept = prep.get_feature_names_out()[l1.named_steps["model"].coef_.ravel() != 0]
print("L1 kept:", list(kept))
pipes["L1, C=0.1"] = l1
keep_row(results, score_row("L1, C=0.1", y_val, l1.predict_proba(X_val)[:, 1]))

- L1 at `C=0.1`: 39 of 51 weights are exactly zero, and validation F1 drops to about 0.276. L2 at the same `C`: none of 51, F1 about 0.408.
- L1 kept 12 encoded columns, among them `duration`, `credit_amount`, `employment` and two `checking_status` values: selection done by the penalty, the third family of 4.1. Here it went too far; a larger `C` is the next thing to try.
- Its row joins `results`: F1 0.276, recall 0.20, cost 255, the worst so far.
- Newer scikit-learn releases (1.8 and later) deprecate `penalty=` in favour of `l1_ratio=1` for L1 and print a FutureWarning on this cell. Colab's scikit-learn 1.6.1 documents `penalty` with no deprecation, so the line is right as written there.
- **Common mistake:** asking the default solver for L1. `LogisticRegression(penalty="l1")` with `lbfgs` raises an error; L1 needs `liblinear` or `saga`.

**In an interview:** "L1 penalises the absolute weights and drives some to exactly zero, so it doubles as feature selection; L2 penalises squared weights and only shrinks them. In scikit-learn's logistic regression, L1 needs a solver that supports it, such as `liblinear`."

### 6.3 L2 in Keras

In Keras the penalty goes on each layer: `kernel_regularizer=tf.keras.regularizers.l2(0.01)` adds 0.01 times the sum of the layer's squared weights (its **kernel**) to the loss. Two networks read the 51 encoded columns from 2.4's `prep`:

- `credit_net`: `Input` of the encoded width, `Dense(32, activation="relu")`, `Dense(1, activation="sigmoid")`, Adam, `binary_crossentropy`.
- `credit_net_l2`: the same, with the L2 penalty on both `Dense` layers.

Both train for 100 epochs with `batch_size=32`, with the encoded validation rows as `validation_data`.

**Predict first:** how many extra weights does the L2 penalty add to the network?

In [ ]:
X_tr_enc, X_val_enc = prep.transform(X_tr), prep.transform(X_val)   # 2.4's prep, fitted on X_tr

def make_credit_net(l2=None):
    """One hidden layer of 32 ReLU units and a sigmoid output; with l2, both layers carry an L2 penalty."""
    reg = tf.keras.regularizers.l2(l2) if l2 else None
    net = tf.keras.Sequential([
        tf.keras.Input(shape=(X_tr_enc.shape[1],)),          # the encoded width
        tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=reg),
        tf.keras.layers.Dense(1, activation="sigmoid", kernel_regularizer=reg)])
    net.compile(optimizer="adam", loss="binary_crossentropy")
    return net

tf.keras.utils.set_random_seed(RANDOM_STATE)
credit_net = make_credit_net()
tf.keras.utils.set_random_seed(RANDOM_STATE)
credit_net_l2 = make_credit_net(l2=0.01)
print(f"input width {X_tr_enc.shape[1]}; parameters {credit_net.count_params():,} and {credit_net_l2.count_params():,}")

- 1,697 parameters each: 51 × 32 + 32 in the hidden layer and 32 + 1 in the output. The penalty adds no weight, only a term in the loss.

The fits take about 15 seconds each in our run:

In [ ]:
histories = {}
for name, net in [("credit_net", credit_net), ("credit_net_l2", credit_net_l2)]:
    start = time.time()
    histories[name] = net.fit(X_tr_enc, y_tr, epochs=100, batch_size=32,
                              validation_data=(X_val_enc, y_val), verbose=0)
    print(f"{name}: 100 epochs in {time.time() - start:.0f} s, final validation loss "
          f"{histories[name].history['val_loss'][-1]:.3f}")

`plot_history` draws training and validation loss for each network:

In [ ]:
plot_history(histories["credit_net"], "loss", "credit_net: no penalty")
plot_history(histories["credit_net_l2"], "loss", "credit_net_l2: the loss includes the L2 penalty")

- `credit_net`'s training loss keeps falling, to about 0.20 by epoch 100, while its validation loss is lowest (about 0.56) around epoch 16 and then turns up, to about 0.63: the network is overfitting its 600 rows.
- `credit_net_l2`'s curves stay closer together, about 0.48 and 0.62 at epoch 100. But Keras adds the penalty to the loss it reports, so the two networks' losses are not on the same footing: the L2 loss is higher by construction (it starts above 1.0). Compare them on validation F1, recall and cost instead.

In [ ]:
for name, net in [("credit_net", credit_net), ("credit_net_l2", credit_net_l2)]:
    keep_row(results, score_row(name, y_val, net.predict(X_val_enc, verbose=0).ravel()))   # sigmoid output at 0.5
pd.DataFrame(results).round(3)

- At threshold 0.5, in our run `credit_net` scored F1 about 0.486, recall 0.433 and cost 191, and `credit_net_l2` about 0.396, 0.350 and 220. Here the penalty at 0.01 cost more than it saved; like `C`, its strength is a value to compare on validation rows.
- `credit_net` has the lowest cost at 0.5 so far. The kept model in 6.4 is chosen among the scikit-learn pipelines, so that one object carries every preparation step; the networks' rows are a comparison.
- **Common mistake:** comparing the two loss curves and calling the L2 network worse, or better. Its loss includes the penalty; compare validation F1, recall and cost.

**In an interview:** "In Keras I add L2 with `kernel_regularizer` on each layer. The penalty is part of the reported loss, so I compare a regularised network with an unregularised one on validation metrics, not on the loss curve."

### 6.4 Cross-validation

Every score so far rests on one split of 200 validation rows with 60 defaulters. **k-fold cross-validation** splits the rows into k folds, trains on k - 1 of them and scores the one left out, k times, and reports the mean and the spread of the k scores.

`StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)` keeps the 30% share of defaulters in every fold; its `random_state` has effect only with `shuffle=True`. `cross_val_score` refits the whole pipeline inside each fold, on the 800 `X_trval` rows, so each fold holds 160 rows, 48 of them defaulters.

**Predict first:** the same pipeline, five different held-out folds. Will the five F1 scores be close together or spread out?

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)   # Try changing it: n_splits=10
scores = cross_val_score(make_pipe(), X_trval, y_trval, cv=cv, scoring="f1")   # refits the pipeline in every fold
print(f"{len(scores)} fold scores: {scores.round(2)}")
print(f"mean {scores.mean():.2f}, standard deviation {scores.std():.2f}, "
      f"from {scores.min():.2f} to {scores.max():.2f}")

- Five scores: 0.56, 0.62, 0.47, 0.30 and 0.57. Mean about 0.50, standard deviation about 0.11.
- The single validation F1 of 0.425 was one draw from that spread. Report the mean and the spread: about 0.50, give or take 0.11.
- Try changing `n_splits=5` to `n_splits=10` and rerun this cell only: ten scores from about 0.24 to 0.65, mean about 0.49, standard deviation about 0.12. More, smaller folds give noisier scores. Put it back to 5 and rerun before the next cell.

**The kept model.** The next cell picks, from `results`, the scikit-learn pipeline with the lowest `val_cost`, refits it on `X_tr` as `best_pipe`, and tunes a threshold for it on its own validation probabilities as `best_threshold`. A threshold belongs to the model it was tuned for, so it is always tuned again for the model that is kept.

In [ ]:
best_row = min((r for r in results if r["model"] in pipes), key=lambda r: r["val_cost"])   # scikit-learn rows only
best_name = best_row["model"]
best_pipe = clone(pipes[best_name]).fit(X_tr, y_tr)       # the kept model, refitted on the training rows
best_threshold, best_cost = best_cost_threshold(y_val, best_pipe.predict_proba(X_val)[:, 1])   # tuned for IT
print(f"kept: {best_name} (val_cost {best_row['val_cost']} at threshold 0.5)")
print(f"best_threshold {best_threshold}: validation cost {best_cost}")

- Kept: `clip`, with a cost of 204 at 0.5 (tied with `log`; the first is kept).
- `best_threshold` is 0.1, with a validation cost of 99: the same as 5.4, because 5.4 swept the same pipeline. Had a later row taken the lead, this cell would have found that model's own threshold.
- **Common mistake:** carrying a threshold over from another model. Each model's probabilities sit differently, so a threshold is tuned on validation rows for the model it is used with.

**In an interview:** "Stratified k-fold cross-validation trains k times, each time scoring a different fold that keeps the class balance, and I report the mean and the standard deviation. On small data one validation split can be lucky or unlucky, and the spread says how far one number can be trusted."

### 6.5 Choosing C with GridSearchCV

*Going deeper.* The idea to keep: choosing a setting on the folds makes their score optimistic, so the test set still decides once.

`GridSearchCV` tries each value of `C` with cross-validation and keeps the best. `model__C` names the `C` of the pipeline step called `model`. It reports `best_params_` and `best_score_`, the mean cross-validated score of the winner, and refits the winner on all the rows it was given. This cell only compares; it never replaces `best_pipe`.

In [ ]:
grid_search = GridSearchCV(make_pipe(), {"model__C": [0.01, 0.1, 1, 10, 100]}, scoring="f1", cv=cv)
grid_search.fit(X_trval, y_trval)                          # the C of the step named "model"
print(f"chosen: {grid_search.best_params_}, mean cross-validated F1 {grid_search.best_score_:.3f}")
print(f"best_pipe ({best_name}) uses C={best_pipe.named_steps['model'].C}; best_pipe is not replaced")

- The grid chose `C=10`, with a mean cross-validated F1 of about 0.506 on the 800 rows. The same folds chose and scored it, so that number is optimistic.
- `best_pipe` keeps `C=1.0`; the difference between `C=1` and `C=10` in 6.1 was about 0.01 F1, inside the spread of 6.4.

**Before you move on**, check that you can:

- say whether a larger `C` means more or less regularisation, and what happens at a very small `C`;
- write an L1 logistic regression that runs, and say why L1 zeroes weights and L2 does not;
- add an L2 penalty to a Keras `Dense` layer, and say why its loss curve is not compared directly;
- run stratified 5-fold cross-validation and report the mean and the spread.

## 7. The test set once, and your project

### 7.1 The kept pipeline and threshold on X_test

`X_test` has waited since 1.1. Every choice (features, encoding, outlier handling, `C`, the threshold) was made on training or validation rows. The cell scores `best_pipe` at `best_threshold` on the 200 test rows, once. A test number read more than once becomes another validation number and stops being honest.

**Predict first:** at its threshold of 0.1, `best_pipe` had a validation recall of about 0.97 (58 of 60) and a cost of 99. Will the test numbers match them exactly?

In [ ]:
test_pred = (best_pipe.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)   # X_test, once
print(f"{best_name} at threshold {best_threshold} on the {len(y_test)} test rows ({y_test.sum()} defaulters):")
print(f"F1 {f1_score(y_test, test_pred):.3f}, recall {recall_score(y_test, test_pred):.3f}, "
      f"total cost {total_cost(y_test, test_pred)}")
print(confusion_matrix(y_test, test_pred))
print(f"'always good' on the test rows would cost {5 * y_test.sum()}")

- On the test rows: F1 about 0.521, recall about 0.933 (56 of 60 defaulters refused) and a total cost of 119: 4 defaulters approved (20) and 99 good applicants refused (99).
- Close to validation, not equal: a different 200 rows, and the spread of 6.4 says how far one number can move. "Always good" would cost 300 on these rows.
- Under these costs the kept rule refuses most applicants, good ones included. That is what a 5-to-1 cost buys with a model this weak; a lender with other costs would get another threshold.
- **Common mistake:** going back to earlier cells after seeing the test number to "improve" it. That turns the test set into validation data. Report the number as it is.

**In an interview:** "I use the test set once, at the end, for the model and the threshold I kept, after every choice was made on training or validation rows. If I tune anything after seeing the test score, it is no longer a test score."

### 7.2 A checklist for your own notebook

Four things to bring into your own project notebook, one at a time:

- **The metric and its reason.** Name the metric, and say which mistake costs more for your problem and who would decide that.
- **A threshold, if your model answers yes or no.** Tune it on validation rows for the model you keep, never carried over from another model. A regression model has no threshold.
- **One regularisation as an improvement attempt.** L2 or L1 in scikit-learn, or `kernel_regularizer` in Keras, measured on the same validation data as the model before it: one change, one new row.
- **Cross-validation when the data is small.** Report the mean and the spread rather than one validation number.

If your dataset holds a column a decision may not rest on, leave it and anything built from it out, and say so in the notebook, as 1.1 did.

**Before you move on**, check that you can:

- say why the test set is used once, and what it is used for;
- go through the four checklist items for your own project and name the one you will do first.

## Recap

| Idea | What it is for | Watch out for |
|---|---|---|
| Columns a lender may not use | keeping decisions off sex, marital status, national origin and age | drop them, and features built from them, before any score is read |
| A new feature (ratio, band) | handing a model what it cannot build itself | judge it on validation rows; ask whether it exists at decision time |
| One-hot, ordinal, frequency encoding | turning categories into numbers by what the values mean and how many there are | write the real order out for ordinal; one-hot is wrong for hundreds of values |
| `ColumnTransformer` and `Pipeline` | every preparation step and the model as one object | fit on the training rows only; never `fit_transform` validation rows |
| Median fill and indicator | blanks handled from the training rows | the indicator helps only when blanks carry meaning |
| 1.5 IQR fence; keep, clip or log | finding and taming extreme values | fences from `X_tr`; error or real; trees hardly care |
| Mutual information, permutation importance | ranking features without and with a model | permutation on validation rows; correlated columns share importance |
| Metric and threshold from costs | reporting what the decision owner pays for | 70% accuracy can mean nothing; tune the threshold for the kept model |
| L2 and L1, `C` | penalising large weights; L1 also selects | smaller `C` is stronger; a Keras L2 loss includes its penalty |
| Stratified k-fold | a mean and a spread instead of one number | `shuffle=True` for `random_state` to matter |
| The test set once | an honest final number | no tuning after looking at it |

**Quick check**

1. A colleague adds `age` back to the credit model because it raises validation F1 by 0.02. What do you say?
2. A column has 800 distinct values, such as a product code. Why is one-hot the wrong choice, and what would you use instead?
3. Blanks in a column fall mostly on applicants who later defaulted. Median fill, or median fill plus an indicator, and why?
4. What does putting the preparation and the model in one `Pipeline` prevent?
5. A defaulter approved costs 5 and a good applicant refused costs 1. Where should the threshold be, and on which rows do you choose it?
6. Does a larger `C` mean more or less regularisation, and what do you report from five cross-validation scores?

<details>
<summary>Answers (say yours aloud first)</summary>

1. Age is a protected ground under the Canadian Human Rights Act, and Regulation B in the US allows it only in a narrow use that needs legal review. A better score does not make a column allowed, so it stays out, and so does any stand-in for it.
2. One-hot would add 800 mostly-zero columns and keep meeting codes it never saw. I would group rare values with `min_frequency` or `max_categories`, use frequency encoding (one column of shares from the training rows), or use target encoding, cross-fitted so the label does not leak.
3. Median plus an indicator: the fact that the value is missing carries information, and the indicator column lets the model use it. In our run it lifted validation F1 from about 0.41 to 0.59 on such a column, and did nothing useful when the blanks were random.
4. Leakage in the preparation: every fill, scale and category list is learned from the training rows only, and cross-validation refits all of it inside each fold, so no held-out row shapes the preparation.
5. Near 1 / (1 + 5), about 0.17, checked with a sweep of the validation probabilities of the model I keep (0.10 in our run). Never on the test rows, never on the training rows, and never carried over from another model.
6. Less: `C` is the inverse of the strength, so a smaller `C` regularises more. From five fold scores I report the mean and the standard deviation, about 0.50 give or take 0.11 for the baseline here.

</details>